# Homework 06 - Trees
The goal of this homework is to predict the fuel efficiency of a car using a decision tree regression model.

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import export_text

## Dataset
This homework will be using the fuel efficiency dataset.

In [3]:
fe_url = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
fe_df = pd.read_csv(fe_url)
fe_df.columns = fe_df.columns.str.lower().str.replace(' ', '_')
fe_df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


## Data preparation
Replace any missing values with `0.0`.
Split the data into train/val/test (60%/20%/20%).

In [7]:
fe_df.isna().sum()

engine_displacement      0
num_cylinders          482
horsepower             708
vehicle_weight           0
acceleration           930
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors              502
fuel_efficiency_mpg      0
dtype: int64

In [9]:
fe_df = fe_df.fillna(0.0)

In [10]:
fe_df.isna().sum()

engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
fuel_efficiency_mpg    0
dtype: int64

In [11]:
full_train_df, test_df = train_test_split(fe_df, test_size=0.2, random_state=1)
train_df, val_df = train_test_split(full_train_df, test_size=0.25, random_state=1)
(len(train_df), len(val_df), len(test_df))

(5822, 1941, 1941)

## Question 1
With a `max_depth=1`, what feature is used for splitting the data?

In [41]:
def train_dtree(tr_df, depth):
    train_dict = tr_df.drop('fuel_efficiency_mpg', axis=1).to_dict(orient='records')
    train_y = tr_df.fuel_efficiency_mpg.values

    
    dv = DictVectorizer(sparse=False)
    train_X = dv.fit_transform(train_dict)
    
    dt = DecisionTreeRegressor(max_depth=depth)
    dt.fit(train_X, train_y)

    return dt, dv

In [42]:
train_df.fuel_efficiency_mpg.values

array([15.3014754 , 15.33121466, 15.33667895, ..., 15.18828665,
       17.3967514 , 16.16090373], shape=(5822,))

In [44]:
dtree, dvect = train_dtree(train_df, 1)
print(export_text(dtree, feature_names=list(dvect.get_feature_names_out())))

|--- vehicle_weight <= 3022.11
|   |--- value: [16.88]
|--- vehicle_weight >  3022.11
|   |--- value: [12.94]



## Question 2
After training a random forest regressor, what is the RMSE on the validation data?

In [40]:
def rmse(y, y_pred):
    err = y_pred - y
    sqrd_err = err ** 2
    mean_sqrd_err = sqrd_err.mean()
    return np.sqrt(mean_sqrd_err)

In [51]:
def train_and_test_rforest(tr_df, v_df):
    train_dict = tr_df.drop('fuel_efficiency_mpg', axis=1).to_dict(orient='records')
    train_y = tr_df.fuel_efficiency_mpg.values

    val_dict = v_df.drop('fuel_efficiency_mpg', axis=1).to_dict(orient='records')
    val_y = v_df.fuel_efficiency_mpg.values
    
    dv = DictVectorizer(sparse=False)
    train_X = dv.fit_transform(train_dict)
    val_X = dv.transform(val_dict)

    rf = RandomForestRegressor(n_estimators=10, random_state=1)
    rf.fit(train_X, train_y)

    pred_y = rf.predict_proba(val_X)[:, 1]

    return rmse(val_y, pred_y)

In [52]:
train_and_test_rforest(train_df, val_df)

AttributeError: 'RandomForestRegressor' object has no attribute 'predict_proba'